# Surface Code Threshold: Multi-DEM PyMatching Comparison

This notebook estimates the circuit-level depolarizing noise threshold for the surface code
by comparing **5 decoder/DEM variants**, all using PyMatching MWPM:

1. **PECOS API** -- `SurfaceDecoder(decoder_type="pymatching")` (uses PECOS raw DEM internally)
2. **PM + PECOS raw** -- PyMatching direct API with `dem.to_string()`
3. **PM + PECOS decomposed** -- PyMatching direct API with `dem.to_string_decomposed()`
4. **PM + Stim raw** -- PyMatching direct API with `circuit.detector_error_model(decompose_errors=False)`
5. **PM + Stim decomposed** -- PyMatching direct API with `circuit.detector_error_model(decompose_errors=True)`

**Setup:**
- Distances 9, 11, 13, 15 (avoiding finite-size effects)
- Uniform depolarizing noise (p1 = p2 = p_meas = p_prep = p)
- 2*d syndrome rounds per distance
- Selene simulation with Stim backend

**Expected results:**
- PECOS API and PM+PECOS raw give identical LERs (same DEM, same decoder)
- Raw vs decomposed give very similar LERs per DEM source
- All variants show correct threshold behavior (~0.5-1% for circuit-level depolarizing)

In [ ]:
import time
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pymatching
import stim
from pecos.compilation_pipeline import compile_guppy_to_hugr
from pecos.guppy.surface import get_num_qubits, make_surface_code
from pecos.misc.threshold_curve import func, func6, threshold_fit
from pecos.qec import DagFaultAnalyzer, DemBuilder
from pecos.qec.surface import NoiseModel, SurfaceDecoder, SurfacePatch
from pecos.qec.surface.circuit_builder import _extract_measurement_order, generate_tick_circuit_from_patch
from pecos.qec.surface.decode import build_stim_circuit_from_patch
from selene_sim import DepolarizingErrorModel, SimpleRuntime, Stim, build

## Configuration

In [ ]:
DISTANCES = [9, 11, 13, 15]
NUM_SHOTS = 1000
ERROR_RATES = [0.001, 0.002, 0.003, 0.005, 0.007, 0.01]
VARIANTS = ["pecos_api", "pm_pecos_raw", "pm_pecos_decomp", "pm_stim_raw", "pm_stim_decomp"]

## Simulation Helpers

Circuits are compiled once per distance. Shots are simulated once per (distance, error rate)
and reused across all decoder variants.

In [ ]:
def compile_circuit(distance: int, num_rounds: int, basis: str):
    """Compile a surface code circuit once, return (instance, num_qubits)."""
    num_qubits = get_num_qubits(distance)
    prog = make_surface_code(distance=distance, num_rounds=num_rounds, basis=basis)
    hugr_bytes = compile_guppy_to_hugr(prog)
    instance = build(hugr_bytes, name=f"surface_d{distance}_{basis}_{num_rounds}r")
    return instance, num_qubits


def run_shots(instance, num_qubits: int, num_shots: int, error_model: Any):
    """Run shots and collect syndrome/final data."""
    all_shots = []
    for shot_results in instance.run_shots(
        simulator=Stim(),
        n_qubits=num_qubits,
        n_shots=num_shots,
        error_model=error_model,
        runtime=SimpleRuntime(),
        n_processes=1,
    ):
        synx_list = []
        synz_list = []
        final = None
        for name, values in shot_results:
            vals = list(values)
            if name == "synx":
                synx_list.append(np.array(vals, dtype=np.uint8))
            elif name == "synz":
                synz_list.append(np.array(vals, dtype=np.uint8))
            elif name == "final":
                final = np.array(vals, dtype=np.uint8)
        if final is not None:
            all_shots.append((synx_list, synz_list, final))
    return all_shots

## DEM Generation

Generate DEMs from both PECOS native (DemBuilder) and Stim pipelines,
each in raw and decomposed form.

In [ ]:
def generate_pecos_dems(patch, num_rounds, noise, basis):
    """Generate PECOS native DEMs (raw and decomposed).

    Uses: TickCircuit -> DagFaultAnalyzer -> DemBuilder -> build()

    Returns:
        (raw_str, decomposed_str)
    """
    tc = generate_tick_circuit_from_patch(patch, num_rounds, basis)
    dag = tc.to_dag_circuit()
    analyzer = DagFaultAnalyzer(dag)
    influence_map = analyzer.build_influence_map()

    detectors_json = tc.get_meta("detectors")
    observables_json = tc.get_meta("observables")
    num_measurements = int(tc.get_meta("num_measurements") or "0")
    measurement_order = _extract_measurement_order(tc)

    builder = DemBuilder(influence_map)
    builder.with_noise(noise.p1, noise.p2, noise.p_meas, noise.p_prep)
    builder.with_num_measurements(num_measurements)
    builder.with_measurement_order(measurement_order)
    builder.with_detectors_json(detectors_json)
    if observables_json:
        builder.with_tracked_paulis_json(observables_json)

    dem = builder.build()
    return dem.to_string(), dem.to_string_decomposed()


def generate_stim_dems(patch, num_rounds, noise, basis):
    """Generate Stim DEMs (raw and decomposed).

    Uses: build_stim_circuit_from_patch -> detector_error_model()

    Returns:
        (raw_dem, decomposed_dem) as stim.DetectorErrorModel objects
    """
    circuit = build_stim_circuit_from_patch(patch, num_rounds, noise, basis)
    raw = circuit.detector_error_model(decompose_errors=False)
    decomposed = circuit.detector_error_model(decompose_errors=True)
    return raw, decomposed

## Detection Event Computation

PECOS and Stim DEMs define detectors in different orders. Detection events
must be computed in the order matching each DEM source.

**PECOS DEM ordering:**
- Z-basis: X-stab rounds 1+ | Z-stab all rounds | Final Z-stab
- X-basis: X-stab all rounds | Z-stab rounds 1+ | Final X-stab

**Stim DEM ordering:**
- Z-basis: Z-stab round 0 | (X-stab + Z-stab) rounds 1+ | Final Z-stab
- X-basis: X-stab round 0 | (X-stab + Z-stab) rounds 1+ | Final X-stab

In [ ]:
def compute_pecos_events_z(synx_list, synz_list, final, geom, num_rounds):
    """Compute detection events in PECOS DEM order for Z-basis."""
    synx = np.array(synx_list, dtype=np.uint8)
    synz = np.array(synz_list, dtype=np.uint8)
    events = []

    # 1. X stabilizer detection events (rounds 1 to num_rounds-1)
    for r in range(1, num_rounds):
        events.extend((synx[r] ^ synx[r - 1]).tolist())

    # 2. Z stabilizer detection events (all rounds)
    events.extend(synz[0].tolist())  # round 0: compare to expected 0
    for r in range(1, num_rounds):
        events.extend((synz[r] ^ synz[r - 1]).tolist())

    # 3. Final round: parity of final data on each Z stabilizer XOR last syndrome
    for stab in geom.z_stabilizers:
        data_parity = sum(int(final[q]) for q in stab.data_qubits) % 2
        last_syn = int(synz[-1][stab.index])
        events.append((data_parity ^ last_syn) & 1)

    return np.array(events, dtype=np.uint8)


def compute_pecos_events_x(synx_list, synz_list, final, geom, num_rounds):
    """Compute detection events in PECOS DEM order for X-basis."""
    synx = np.array(synx_list, dtype=np.uint8)
    synz = np.array(synz_list, dtype=np.uint8)
    events = []

    # 1. X stabilizer detection events (all rounds)
    events.extend(synx[0].tolist())  # round 0: compare to expected 0
    for r in range(1, num_rounds):
        events.extend((synx[r] ^ synx[r - 1]).tolist())

    # 2. Z stabilizer detection events (rounds 1 to num_rounds-1)
    for r in range(1, num_rounds):
        events.extend((synz[r] ^ synz[r - 1]).tolist())

    # 3. Final round: parity of final data on each X stabilizer XOR last syndrome
    for stab in geom.x_stabilizers:
        data_parity = sum(int(final[q]) for q in stab.data_qubits) % 2
        last_syn = int(synx[-1][stab.index])
        events.append((data_parity ^ last_syn) & 1)

    return np.array(events, dtype=np.uint8)


def compute_stim_events_z(synx_list, synz_list, final, geom, num_rounds):
    """Compute detection events in Stim DEM order for Z-basis."""
    synx = np.array(synx_list, dtype=np.uint8)
    synz = np.array(synz_list, dtype=np.uint8)
    events = []

    # 1. Z stabilizer detectors at round 0 (deterministic for Z-basis)
    events.extend(synz[0].tolist())

    # 2. Rounds 1+: X stabilizers then Z stabilizers
    for r in range(1, num_rounds):
        events.extend((synx[r] ^ synx[r - 1]).tolist())
        events.extend((synz[r] ^ synz[r - 1]).tolist())

    # 3. Final Z-stab: last syndrome XOR final data parity
    for stab in geom.z_stabilizers:
        data_parity = sum(int(final[q]) for q in stab.data_qubits) % 2
        last_syn = int(synz[-1][stab.index])
        events.append((data_parity ^ last_syn) & 1)

    return np.array(events, dtype=np.uint8)


def compute_stim_events_x(synx_list, synz_list, final, geom, num_rounds):
    """Compute detection events in Stim DEM order for X-basis."""
    synx = np.array(synx_list, dtype=np.uint8)
    synz = np.array(synz_list, dtype=np.uint8)
    events = []

    # 1. X stabilizer detectors at round 0 (deterministic for X-basis)
    events.extend(synx[0].tolist())

    # 2. Rounds 1+: X stabilizers then Z stabilizers
    for r in range(1, num_rounds):
        events.extend((synx[r] ^ synx[r - 1]).tolist())
        events.extend((synz[r] ^ synz[r - 1]).tolist())

    # 3. Final X-stab: last syndrome XOR final data parity
    for stab in geom.x_stabilizers:
        data_parity = sum(int(final[q]) for q in stab.data_qubits) % 2
        last_syn = int(synx[-1][stab.index])
        events.append((data_parity ^ last_syn) & 1)

    return np.array(events, dtype=np.uint8)

## Decoding

Decode all 5 variants for a batch of shots. DEMs and matchers are created
once per (distance, error_rate) and reused across shots.

In [ ]:
def decode_all_variants(shots, patch, num_rounds, noise, basis):
    """Decode all 5 variants for a batch of shots.

    Returns:
        dict of {variant: {"ler": float, "raw_rate": float, "n": int}}
    """
    geom = patch.geometry
    if basis == "Z":
        logical_qubits = geom.logical_z.data_qubits
        pecos_events_fn = compute_pecos_events_z
        stim_events_fn = compute_stim_events_z
    else:
        logical_qubits = geom.logical_x.data_qubits
        pecos_events_fn = compute_pecos_events_x
        stim_events_fn = compute_stim_events_x

    # Generate all DEMs
    pecos_raw_str, pecos_decomp_str = generate_pecos_dems(patch, num_rounds, noise, basis)
    stim_raw_dem, stim_decomp_dem = generate_stim_dems(patch, num_rounds, noise, basis)

    # Create PyMatching matchers (direct API)
    pm_pecos_raw = pymatching.Matching(stim.DetectorErrorModel(pecos_raw_str))
    pm_pecos_decomp = pymatching.Matching(stim.DetectorErrorModel(pecos_decomp_str))
    pm_stim_raw = pymatching.Matching(stim_raw_dem)
    pm_stim_decomp = pymatching.Matching(stim_decomp_dem)

    # Create PECOS API decoder
    pecos_decoder = SurfaceDecoder(patch, num_rounds=num_rounds, noise=noise, decoder_type="pymatching")

    counts = {v: {"logical_errors": 0, "raw_errors": 0} for v in VARIANTS}
    n = len(shots)

    for synx_list, synz_list, final in shots:
        raw_parity = sum(int(final[q]) for q in logical_qubits) % 2
        if raw_parity != 0:
            for v in VARIANTS:
                counts[v]["raw_errors"] += 1

        # PECOS API
        if basis == "Z":
            is_error, _ = pecos_decoder.decode_memory_z(synx_list, synz_list, final)
        else:
            is_error, _ = pecos_decoder.decode_memory_x(synx_list, synz_list, final)
        if is_error:
            counts["pecos_api"]["logical_errors"] += 1

        # Compute events in both orderings
        pecos_ev = pecos_events_fn(synx_list, synz_list, final, geom, num_rounds)
        stim_ev = stim_events_fn(synx_list, synz_list, final, geom, num_rounds)

        final_parity = sum(int(final[q]) for q in logical_qubits) % 2

        # PM + PECOS raw
        pred = pm_pecos_raw.decode(pecos_ev)
        if (final_parity + pred[0]) % 2 != 0:
            counts["pm_pecos_raw"]["logical_errors"] += 1

        # PM + PECOS decomposed
        pred = pm_pecos_decomp.decode(pecos_ev)
        if (final_parity + pred[0]) % 2 != 0:
            counts["pm_pecos_decomp"]["logical_errors"] += 1

        # PM + Stim raw
        pred = pm_stim_raw.decode(stim_ev)
        if (final_parity + pred[0]) % 2 != 0:
            counts["pm_stim_raw"]["logical_errors"] += 1

        # PM + Stim decomposed
        pred = pm_stim_decomp.decode(stim_ev)
        if (final_parity + pred[0]) % 2 != 0:
            counts["pm_stim_decomp"]["logical_errors"] += 1

    return {
        v: {
            "ler": counts[v]["logical_errors"] / n if n > 0 else 0,
            "raw_rate": counts[v]["raw_errors"] / n if n > 0 else 0,
            "n": n,
        }
        for v in VARIANTS
    }

## Z Basis Threshold Sweep (2*d rounds)

In [ ]:
results_z = {v: [] for v in VARIANTS}

print("=== Z Basis Threshold Sweep (2*d rounds) ===")

for d in DISTANCES:
    num_rounds = 2 * d
    t_compile = time.time()
    instance, nq = compile_circuit(d, num_rounds, "Z")
    print(f"\nd={d}, {num_rounds} rounds (compile: {time.time()-t_compile:.1f}s)")

    patch = SurfacePatch.create(d)

    for p in ERROR_RATES:
        error_model = DepolarizingErrorModel(p_1q=p, p_2q=p, p_meas=p, p_init=p)
        noise = NoiseModel(p1=p, p2=p, p_meas=p, p_prep=p)

        t0 = time.time()
        shots = run_shots(instance, nq, NUM_SHOTS, error_model)
        t_sim = time.time() - t0

        t1 = time.time()
        variant_results = decode_all_variants(shots, patch, num_rounds, noise, "Z")
        t_dec = time.time() - t1

        parts = [f"  p={p:.3f} (sim:{t_sim:.1f}s dec:{t_dec:.1f}s):"]
        for v in VARIANTS:
            ler = variant_results[v]["ler"]
            raw = variant_results[v]["raw_rate"]
            results_z[v].append((d, p, ler, raw))
            parts.append(f"{v}={ler:.4f}")

        print("  ".join(parts))

## X Basis Threshold Sweep (2*d rounds)

In [ ]:
results_x = {v: [] for v in VARIANTS}

print("=== X Basis Threshold Sweep (2*d rounds) ===")

for d in DISTANCES:
    num_rounds = 2 * d
    t_compile = time.time()
    instance, nq = compile_circuit(d, num_rounds, "X")
    print(f"\nd={d}, {num_rounds} rounds (compile: {time.time()-t_compile:.1f}s)")

    patch = SurfacePatch.create(d)

    for p in ERROR_RATES:
        error_model = DepolarizingErrorModel(p_1q=p, p_2q=p, p_meas=p, p_init=p)
        noise = NoiseModel(p1=p, p2=p, p_meas=p, p_prep=p)

        t0 = time.time()
        shots = run_shots(instance, nq, NUM_SHOTS, error_model)
        t_sim = time.time() - t0

        t1 = time.time()
        variant_results = decode_all_variants(shots, patch, num_rounds, noise, "X")
        t_dec = time.time() - t1

        parts = [f"  p={p:.3f} (sim:{t_sim:.1f}s dec:{t_dec:.1f}s):"]
        for v in VARIANTS:
            ler = variant_results[v]["ler"]
            raw = variant_results[v]["raw_rate"]
            results_x[v].append((d, p, ler, raw))
            parts.append(f"{v}={ler:.4f}")

        print("  ".join(parts))

## Results Visualization

In [ ]:
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
markers = ["o", "s", "^", "D"]

variant_labels = {
    "pecos_api": "PECOS API",
    "pm_pecos_raw": "PM+PECOS raw",
    "pm_pecos_decomp": "PM+PECOS decomp",
    "pm_stim_raw": "PM+Stim raw",
    "pm_stim_decomp": "PM+Stim decomp",
}

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

for row, (basis_results, basis_label) in enumerate([(results_z, "Z"), (results_x, "X")]):
    # One subplot per variant
    for col, v in enumerate(VARIANTS[:3]):
        ax = axes[row, col]
        for i, d in enumerate(DISTANCES):
            d_dec = [(p, dec) for (dd, p, dec, raw) in basis_results[v] if dd == d]
            if d_dec:
                ps, lers = zip(*d_dec, strict=False)
                lers_plot = [max(val, 1e-4) for val in lers]
                ax.plot(ps, lers_plot, f"{markers[i]}-", color=colors[i],
                        label=f"d={d}", markersize=8, linewidth=2)

        ax.set_xlabel("Physical Error Rate", fontsize=11)
        ax.set_ylabel("Logical Error Rate", fontsize=11)
        ax.set_title(f"{basis_label} Basis: {variant_labels[v]}", fontsize=12)
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3, which="both")
        ax.set_ylim(1e-4, 1.0)

plt.tight_layout()
plt.show()

# Second figure for the remaining 2 variants + combined comparison
fig2, axes2 = plt.subplots(2, 3, figsize=(20, 12))

for row, (basis_results, basis_label) in enumerate([(results_z, "Z"), (results_x, "X")]):
    # Stim variants
    for col, v in enumerate(VARIANTS[3:]):
        ax = axes2[row, col]
        for i, d in enumerate(DISTANCES):
            d_dec = [(p, dec) for (dd, p, dec, raw) in basis_results[v] if dd == d]
            if d_dec:
                ps, lers = zip(*d_dec, strict=False)
                lers_plot = [max(val, 1e-4) for val in lers]
                ax.plot(ps, lers_plot, f"{markers[i]}-", color=colors[i],
                        label=f"d={d}", markersize=8, linewidth=2)

        ax.set_xlabel("Physical Error Rate", fontsize=11)
        ax.set_ylabel("Logical Error Rate", fontsize=11)
        ax.set_title(f"{basis_label} Basis: {variant_labels[v]}", fontsize=12)
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3, which="both")
        ax.set_ylim(1e-4, 1.0)

    # Combined comparison at largest distance
    ax = axes2[row, 2]
    d_max = max(DISTANCES)
    variant_colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
    variant_markers = ["o", "s", "^", "D", "v"]
    for vi, v in enumerate(VARIANTS):
        d_dec = [(p, dec) for (dd, p, dec, raw) in basis_results[v] if dd == d_max]
        if d_dec:
            ps, lers = zip(*d_dec, strict=False)
            lers_plot = [max(val, 1e-4) for val in lers]
            ax.plot(ps, lers_plot, f"{variant_markers[vi]}-", color=variant_colors[vi],
                    label=variant_labels[v], markersize=8, linewidth=2)

    ax.set_xlabel("Physical Error Rate", fontsize=11)
    ax.set_ylabel("Logical Error Rate", fontsize=11)
    ax.set_title(f"{basis_label} Basis: All variants (d={d_max})", fontsize=12)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, which="both")
    ax.set_ylim(1e-4, 1.0)

plt.tight_layout()
plt.show()

## Threshold Estimation

The threshold is estimated using two methods:

1. **Curve crossing**: Find where logical error rate curves for consecutive distances
   cross. Below threshold, increasing distance reduces the LER; above threshold, it
   increases.

2. **Universal scaling fit** (arXiv:quant-ph/0207088): Fit the scaling ansatz
   $p_L = f\bigl((p - p_{th}) \cdot d^{1/\nu_0}\bigr)$ using `pecos.misc.threshold_curve`.
   The fitted parameter $p_{th}$ is the threshold.

In [ ]:
def estimate_threshold_crossings(results, distances):
    """Estimate threshold from crossings between consecutive distance curves."""
    crossings = []

    for idx in range(len(distances) - 1):
        d1 = distances[idx]
        d2 = distances[idx + 1]

        pts1 = {p: ler for (d, p, ler, _) in results if d == d1}
        pts2 = {p: ler for (d, p, ler, _) in results if d == d2}

        common_ps = sorted(set(pts1.keys()) & set(pts2.keys()))
        if len(common_ps) < 2:
            continue

        for j in range(len(common_ps) - 1):
            p_lo = common_ps[j]
            p_hi = common_ps[j + 1]
            diff_lo = pts1[p_lo] - pts2[p_lo]
            diff_hi = pts1[p_hi] - pts2[p_hi]

            if diff_lo * diff_hi < 0:
                t = diff_lo / (diff_lo - diff_hi)
                p_cross = p_lo + t * (p_hi - p_lo)
                crossings.append((d1, d2, p_cross))

    return crossings


def estimate_threshold_fit(results, distances, fit_func=func, p0=None):
    """Estimate threshold using universal scaling fit."""
    filtered = [(d, p, ler) for (d, p, ler, _) in results if ler > 0 and d in distances]
    if len(filtered) < 5:
        return None

    plist = np.array([p for _, p, _ in filtered])
    dlist = np.array([float(d) for d, _, _ in filtered])
    plog = np.array([ler for _, _, ler in filtered])

    if p0 is None:
        p0 = [0.5, 0.005] if fit_func == func6 else [0.005, 1.5, 0.5, 1.0, 0.0]

    try:
        popt, stdev = threshold_fit(plist, dlist, plog, fit_func, p0)
    except (RuntimeError, ValueError) as e:
        print(f"  Fit failed: {e}")
        return None
    else:
        return popt, stdev


# === Build comparison table ===
print("=== Threshold Comparison ===")
print()
header = f"{'Variant':<22} | {'Z crossing':>10} | {'Z fit (func6)':>16} | {'X crossing':>10} | {'X fit (func6)':>16}"
print(header)
print("-" * len(header))

for v in VARIANTS:
    row = f"{variant_labels[v]:<22}"

    for basis_results in [results_z, results_x]:
        data = basis_results[v]

        # Crossing estimate
        crossings = estimate_threshold_crossings(data, DISTANCES)
        if crossings:
            avg_cross = np.mean([c[2] for c in crossings])
            row += f" | {avg_cross:>10.4f}"
        else:
            row += f" | {'N/A':>10}"

        # Fit estimate
        result = estimate_threshold_fit(data, DISTANCES, func6, p0=[0.5, 0.005])
        if result is not None:
            popt, stdev = result
            row += f" | {popt[1]:>7.4f}+/-{stdev[1]:<5.4f}"
        else:
            row += f" | {'N/A':>16}"

    print(row)

print()

# === Detailed crossing info ===
print("\n=== Detailed Curve Crossings ===")
for v in VARIANTS:
    print(f"\n--- {variant_labels[v]} ---")
    for basis, basis_results in [("Z", results_z[v]), ("X", results_x[v])]:
        crossings = estimate_threshold_crossings(basis_results, DISTANCES)
        print(f"  {basis} basis:")
        if crossings:
            for d1, d2, p_cross in crossings:
                print(f"    d={d1}/d={d2} crossing at p ~ {p_cross:.4f}")
            avg = np.mean([c[2] for c in crossings])
            print(f"    Average: p ~ {avg:.4f}")
        else:
            print("    No crossings found.")

## Summary

This notebook estimates the circuit-level depolarizing noise threshold for the surface code
by comparing 5 decoder/DEM variants, all using PyMatching MWPM.

**Setup:**
- Uniform depolarizing noise: p1 = p2 = p_meas = p_prep = p
- 2*d syndrome rounds per distance
- Distances 9-15 to minimize finite-size effects

**Expected observations:**
- **PECOS API = PM+PECOS raw**: Identical LERs (same DEM + same decoder)
- **Raw vs decomposed**: Nearly identical per DEM source
- **PECOS vs Stim**: May differ slightly due to different circuit/noise model details
- All variants show correct threshold behavior (~0.5-1% for circuit-level depolarizing)